In [2]:
import pandas as pd
import scanpy as sc
import numpy as np
import anndata as ad
import os
import colorcet as cc
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
import harmonypy as hm
import scrublet as scr
import traceback
from collections import Counter
from sklearn.decomposition import PCA
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns
import re
from matplotlib.patches import Patch
from PIL import Image

# 42 is for TrueType fonts (editable text in Illustrator)
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
%matplotlib inline

# Define working paths
working_dic = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_control/"
os.chdir(working_dic)

# --- SCANPY SETTINGS ---
sc.settings.verbosity = 1
sc.logging.print_header()

# Professional plotting parameters
# sc.set_figure_params( dpi=100,  dpi_save=300,  vector_friendly=True, format='pdf',  transparent=True, facecolor='white' )

# --- CUSTOM COLOR SCHEMES ---
dimplot_color = mcolors.LinearSegmentedColormap.from_list("custom_cmap", ["#E5E5E5", "#4C4DB7"])

COND_ORDER = [
    "Control",
    "Prematurely-terminated",
    "3month-protocol",
    "6month-protocol",
    "PCMV/PRV",
    "Rejection",
]

COND_PALETTE = {
    "Control": "#EFE48E",
    "3month-protocol": "skyblue",
    "6month-protocol": "#08528f",
    "Prematurely-terminated": "#C9B8D9",
    "PCMV/PRV": "#F0C096",
    "Rejection": "#E09E9E",
}


# Saving pseudobulk counts for edgeR

In [3]:
import os
import scanpy as sc
import pandas as pd
import numpy as np
import scipy.sparse as sp

# configurations
adata_path = '/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/global_object_assembly/objects/global_X_UMAP_Added_PCMV_score_updated_celltype_cellstate_annotated_preclinical_object_RAW_v3.h5ad'

base_out_dir = "/data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_control"

pb_dir = os.path.join(base_out_dir, "pseudobulks")
os.makedirs(pb_dir, exist_ok=True)

print("Loading adata...")
adata = sc.read(adata_path)

# 1. Export metadata
print("Exporting meta_info.csv...")
meta_df = adata.obs[['Sample_ID', 'Condition']].drop_duplicates().reset_index(drop=True)
meta_df.to_csv(os.path.join(base_out_dir, "meta_info.csv"), index=False)

# 2. Generate pseudobulks per cell_type

cell_types = adata.obs['cell_type'].unique()

for ct in cell_types:
    print(f"Aggregating pseudobulk for {ct}...")
    adata_ct = adata[adata.obs['cell_type'] == ct]
    
    # Initialize an empty DataFrame with gene names as index
    pb_df = pd.DataFrame(index=adata_ct.var_names)
    
    for sample in adata_ct.obs['Sample_ID'].unique():
        sample_adata = adata_ct[adata_ct.obs['Sample_ID'] == sample]
        
        # Sum counts across cells for this sample
        if sp.issparse(sample_adata.X):
            sample_sum = np.array(sample_adata.X.sum(axis=0)).flatten()
        else:
            sample_sum = np.array(sample_adata.X.sum(axis=0))
            
        pb_df[sample] = sample_sum

    # Save to CSV (rows = genes, cols = samples)
    pb_df.to_csv(os.path.join(pb_dir, f"{ct}.csv"))

print(f"All pseudobulks successfully saved to {pb_dir}")

Loading adata...
Exporting meta_info.csv...
Aggregating pseudobulk for CM...
Aggregating pseudobulk for NC...
Aggregating pseudobulk for MC...
Aggregating pseudobulk for FB...
Aggregating pseudobulk for EC...
Aggregating pseudobulk for Mye_sus...
Aggregating pseudobulk for Lymph_sus...
Aggregating pseudobulk for Lymph_papio...
Aggregating pseudobulk for Mye_papio...
Aggregating pseudobulk for EC_papio...
All pseudobulks successfully saved to /data/ag_reichart_lindberg/nhadizad/nastaran/projects/xeno/analysis/v1/EdgeR/v1/cell_type/Ref_control/pseudobulks
